In [1]:

import websocket
import json
import csv
import datetime
import os
import pandas as pd
import threading
import time
from sqlalchemy import create_engine
import urllib.parse

In [2]:

WEBSOCKET_URL = "wss://ws.bitget.com/v2/ws/public"
INSTRUMENT_IDS = ["SOLUSDT", "BTCUSDT", "ETHUSDT"]  
CSV_FILE_NAME = "data.csv"


In [3]:

WEBSOCKET_URL = "wss://ws.bitget.com/v2/ws/public"
INSTRUMENT_IDS = ["SOLUSDT", "BTCUSDT", "ETHUSDT"]  
CSV_FILE_NAME = "data.csv"
CANDLESTICK_CSV_FILE = "candlestick_data.csv"
TIMEFRAMES = ["candle1m", "candle5m", "candle15m", "candle1H"]

In [4]:

def tao_file_csv():
    header = [
    "thoi_gian" ,          # Thời gian dữ liệu (ts)
    "timestamp_api",       # Thời gian dữ liệu được lấy từ API (ts ngoài cùng)
    "instId",              # Mã sản phẩm (ví dụ: ETHUSDT)
    "lastPr",              # Giá giao dịch gần nhất
    "bidPr",               # Giá mua cao nhất
    "askPr",               # Giá bán thấp nhất
    "bidSz",               # Khối lượng đặt mua
    "askSz",               # Khối lượng đặt bán
    "open24h",             # Giá mở cửa trong 24h
    "high24h",             # Giá cao nhất 24h
    "low24h",              # Giá thấp nhất 24h
    "change24h",           # % thay đổi trong 24h
    "baseVolume",          # Khối lượng giao dịch trong 24h (theo coin)
    "quoteVolume",         # Khối lượng giao dịch trong 24h (theo tiền tệ)
    "openUtc",             # Giá mở cửa tại UTC+0
    "changeUtc24h",        # Thay đổi tại UTC+0
    "action"               # snapshot (loại push)
]

    
    if not os.path.exists(CSV_FILE_NAME) or os.path.getsize(CSV_FILE_NAME) == 0:
        with open(CSV_FILE_NAME, mode='w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(header)
        print(f"Đã tạo file CSV với {len(header)} cột: {CSV_FILE_NAME}")
    else:
        print(f"File CSV đã tồn tại: {CSV_FILE_NAME}")

tao_file_csv()

Đã tạo file CSV với 17 cột: data.csv


In [5]:

def tao_file_candlestick_csv():
    candlestick_header = [
        "thoi_gian_he_thong",      # Thời gian ghi nhận từ hệ thống
    "start_time",              # Thời gian bắt đầu nến (timestamp từ API)
    "open_price",              # Giá mở cửa
    "high_price",              # Giá cao nhất
    "low_price",               # Giá thấp nhất
    "close_price",             # Giá đóng cửa
    "volume_base",             # Khối lượng base
    "volume_quote",            # Khối lượng quote
    "volume_usdt",             # Khối lượng USDT
    "instId",                  # Sản phẩm (ETHUSDT)
    "timeframe",               # Thời gian khung nến (1m, 5m, ...)
    "action"                   # snapshot hoặc update
]
    
    
    if not os.path.exists(CANDLESTICK_CSV_FILE) or os.path.getsize(CANDLESTICK_CSV_FILE) == 0:
        with open(CANDLESTICK_CSV_FILE, mode='w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(candlestick_header)
        print(f"Đã tạo file Candlestick CSV với {len(candlestick_header)} cột: {CANDLESTICK_CSV_FILE}")
    else:
        print(f"File Candlestick CSV đã tồn tại: {CANDLESTICK_CSV_FILE}")

# Gọi function tạo file
tao_file_candlestick_csv()

Đã tạo file Candlestick CSV với 12 cột: candlestick_data.csv


In [6]:

all_ticker_data = []
all_candlestick_data = []

def on_open_enhanced(ws):
    print("Đã kết nối thành công")
    
    
    for inst_id in INSTRUMENT_IDS:
        ticker_message = {
            "op": "subscribe",
            "args": [
                {    
                    "instType": "SPOT",
                    "channel": "ticker",
                    "instId": inst_id
                }
            ]
        }
        ws.send(json.dumps(ticker_message))
        print(f"Đang theo dõi ticker {inst_id}")
    
    
    for inst_id in INSTRUMENT_IDS:
        for timeframe in TIMEFRAMES:
            candlestick_message = {
                "op": "subscribe",
                "args": [
                    {    
                        "instType": "SPOT",
                        "channel": timeframe,
                        "instId": inst_id
                    }
                ]
            }
            ws.send(json.dumps(candlestick_message))
            print(f"Đang theo dõi candlestick {inst_id} - {timeframe}")
    
    print(f"Hoàn tất subscribe cho {len(INSTRUMENT_IDS)} coins với {len(TIMEFRAMES)} timeframes")

def on_message_enhanced(ws, message_str):
    global all_ticker_data, all_candlestick_data
    
    data = json.loads(message_str)
    
    
    if "arg" in data and "channel" in data["arg"]:
        channel = data["arg"]["channel"]
        
        if channel == "ticker":
            
            all_ticker_data.append(data)
            process_ticker_data(data)
            
        elif channel.startswith("candle"):
            
            all_candlestick_data.append(data)
            process_candlestick_data(data)

def process_ticker_data(data):
    """Xử lý dữ liệu ticker"""
    if "data" in data and data["data"]:
        ticker = data["data"][0]
        
        thoi_gian = datetime.datetime.now().isoformat()  # Thời gian hệ thống ghi nhận
        instId = ticker.get('instId')                   # Mã sản phẩm, ví dụ ETHUSDT
        gia = ticker.get('lastPr')                      # Giá gần nhất
        gia_mua = ticker.get('bidPr')                   # Giá mua cao nhất
        gia_ban = ticker.get('askPr')                   # Giá bán thấp nhất
        khoi_luong_24h = ticker.get('baseVolume')       # Khối lượng giao dịch theo coin (base)
        high24h = ticker.get('high24h')                 # Giá cao nhất trong 24h
        low24h = ticker.get('low24h')                   # Giá thấp nhất trong 24h
        best_purchase_price = ticker.get('bidSz')       # Khối lượng đặt mua
        best_sale_price = ticker.get('askSz')           # Khối lượng đặt bán
        change24h = ticker.get('change24h')             # Tỷ lệ thay đổi trong 24h
        base_volume = ticker.get('baseVolume')          # Trùng với khoi_luong_24h
        quote_volume = ticker.get('quoteVolume')        # Khối lượng giao dịch theo tiền tệ
        open_utc = ticker.get('openUtc')                # Giá mở cửa tại UTC+0
        open24h = ticker.get('open24h')                 # Giá mở cửa 24h
        timestamp_api = ticker.get('ts')                # Thời gian lấy dữ liệu từ API
        action = data.get('action', 'unknown')          # snapshot / unknown nếu không có
                
        
        
        
        print(f"TICKER {instId} | Giá: {gia} | Thay đổi 24h: {change24h}")
        
        
        with open(CSV_FILE_NAME, mode='a', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow([
    thoi_gian, gia, gia_mua, gia_ban, khoi_luong_24h, high24h, low24h, instId,
    best_purchase_price, best_sale_price, change24h,
    base_volume, quote_volume, open_utc, open24h,
    timestamp_api, action
])


def process_candlestick_data(data):
    """Xử lý dữ liệu candlestick"""
    if "data" in data and data["data"]:
        candle = data["data"][0]  
        arg = data.get("arg", {})
        
        
        thoi_gian_he_thong = datetime.datetime.now().isoformat()
        
        # Dữ liệu từ candle array
        start_time = candle[0]  # Timestamp bắt đầu nến
        open_price = candle[1]   # Giá mở cửa
        high_price = candle[2]   # Giá cao nhất
        low_price = candle[3]    # Giá thấp nhất
        close_price = candle[4]  # Giá đóng cửa
        volume_base = candle[5]  # Khối lượng base
        volume_quote = candle[6] # Khối lượng quote
        volume_usdt = candle[7]  # Khối lượng USDT
        
        # Thông tin từ arg và data
        instId = arg.get('instId')
        timeframe = arg.get('channel')
        action = data.get('action', 'unknown')
        
        
        
        try:
            start_time_readable = datetime.datetime.fromtimestamp(int(start_time)/1000).isoformat()
        except:
            start_time_readable = start_time
        
        print(f"CANDLE {instId} {timeframe} | O:{open_price} H:{high_price} L:{low_price} C:{close_price}")
        
       
        with open(CANDLESTICK_CSV_FILE, mode='a', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow([
                thoi_gian_he_thong,  # Thời gian hệ thống
                start_time_readable,  # Thời gian bắt đầu nến (đã convert)
                open_price,          # Giá mở cửa
                high_price,          # Giá cao nhất  
                low_price,           # Giá thấp nhất
                close_price,         # Giá đóng cửa
                volume_base,         # Khối lượng base
                volume_quote,        # Khối lượng quote
                volume_usdt,         # Khối lượng USDT
                instId,              # Sản phẩm (ETHUSDT)
                timeframe,           # Timeframe (candle1m, candle5m, ...)
                action               # Action (snapshot/update)
            ])

def on_error(ws, error):
    print(f"Lỗi: {error}")

def on_close(ws, close_status_code, close_msg):
    print(f"Kết nối đã đóng")

In [7]:

a = 60  # 1 phút
b = a * 60  # 1h
c = b * 24  # 1 ngày

def run_ws_enhanced():
    ws.run_forever(ping_interval=30, ping_timeout=10)

ws = websocket.WebSocketApp(WEBSOCKET_URL,
                          on_open=on_open_enhanced,
                          on_message=on_message_enhanced,
                          on_error=on_error,
                          on_close=on_close)

print("Bắt đầu kết nối đến Bitget (Ticker + Candlestick)...")
print(f"Ticker data sẽ được lưu vào: {CSV_FILE_NAME}")
print(f"Candlestick data sẽ được lưu vào: {CANDLESTICK_CSV_FILE}")
print("Nhấn Ctrl+C để dừng")

ws_thread = threading.Thread(target=run_ws_enhanced)
ws_thread.daemon = True
ws_thread.start()

run_duration = a

try:
    time.sleep(run_duration)
except KeyboardInterrupt:
    print("\nĐã dừng bằng Ctrl+C")

ws.close()
print(f"Đã ngắt kết nối sau {run_duration} giây.")
print(f"Ticker data: {len(all_ticker_data)} messages")
print(f"Candlestick data: {len(all_candlestick_data)} messages")

Bắt đầu kết nối đến Bitget (Ticker + Candlestick)...
Ticker data sẽ được lưu vào: data.csv
Candlestick data sẽ được lưu vào: candlestick_data.csv
Nhấn Ctrl+C để dừng
Đã kết nối thành công
Đang theo dõi ticker SOLUSDT
Đang theo dõi ticker BTCUSDT
Đang theo dõi ticker ETHUSDT
Đang theo dõi candlestick SOLUSDT - candle1m
Đang theo dõi candlestick SOLUSDT - candle5m
Đang theo dõi candlestick SOLUSDT - candle15m
Đang theo dõi candlestick SOLUSDT - candle1H
Đang theo dõi candlestick BTCUSDT - candle1m
Đang theo dõi candlestick BTCUSDT - candle5m
Đang theo dõi candlestick BTCUSDT - candle15m
Đang theo dõi candlestick BTCUSDT - candle1H
Đang theo dõi candlestick ETHUSDT - candle1m
Đang theo dõi candlestick ETHUSDT - candle5m
Đang theo dõi candlestick ETHUSDT - candle15m
Đang theo dõi candlestick ETHUSDT - candle1H
Hoàn tất subscribe cho 3 coins với 4 timeframes
CANDLE SOLUSDT candle1m | O:156.63 H:156.77 L:156.63 C:156.76
CANDLE SOLUSDT candle5m | O:152.99 H:153.14 L:152.52 C:152.53
CANDLE SOL